# 🏠 AQARIA — Notebook COMPLET & CORRIGÉ (Version Finale)
## GPT-2 Fine-tuné · Chatbot + RAG + Évaluateur Premium · Flask API

> **IMPORTANT** : Exécuter les cellules **dans l'ordre** avec **Run All**.
> Ne jamais exécuter la cellule Flask (12b) avant la cellule Routes (12c).

In [1]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 1 — Installation des dépendances
# ════════════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    'flask', 'flask-cors',
    'transformers>=4.35.0',
    'torch', 'psutil', 'requests',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

print('✅ Dépendances installées')

✅ Dépendances installées


In [2]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 2 — Recherche automatique du CSV
# ════════════════════════════════════════════════════════════════
import os, glob

CSV_CANDIDATES = [
    '/kaggle/input/datasets/jmaighofran/daaataaa-csv/data_Immobiler_finale_prete.csv',
    '/kaggle/input/daaataaa-csv/data_Immobiler_finale_prete.csv',
    '/content/drive/MyDrive/data_Immobiler_finale_prete.csv',
    '/content/data_Immobiler_finale_prete.csv',
    './data_Immobiler_finale_prete.csv',
]
CSV_CANDIDATES += glob.glob('/kaggle/input/**/*.csv', recursive=True)

CSV_FOUND = None
for p in CSV_CANDIDATES:
    if os.path.exists(p):
        CSV_FOUND = p
        print(f'  ✅ CSV trouvé : {p}')
        break
if not CSV_FOUND:
    print('  ⚠️  CSV non trouvé. Ajoutez le dataset Kaggle ou uploadez le fichier.')

  ✅ CSV trouvé : /kaggle/input/datasets/jmaighofran/data-immobiler-finale-prete-csv/daaataaa.csv


In [3]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 3 — Configuration & Hyperparamètres
# ════════════════════════════════════════════════════════════════
import json, math, os, random, time, warnings, re, threading
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

CSV_PATH = CSV_FOUND or './data_Immobiler_finale_prete.csv'
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_GPU = (DEVICE == 'cuda')

if USE_GPU:
    BATCH_SIZE, EPOCHS, FP16, GRAD_ACCUM, EST_TIME = 8, 5, True, 1, '~8-12 min'
else:
    BATCH_SIZE, EPOCHS, FP16, GRAD_ACCUM, EST_TIME = 2, 5, False, 4, '~35-45 min'

CONFIG = {
    'csv_path'   : CSV_PATH,
    'output_dir' : './models_aqaria_gpt2_finetuned',
    'model_name' : 'gpt2',
    'max_seq_length'             : 256,
    'learning_rate'              : 3e-4,
    'num_train_epochs'           : EPOCHS,
    'per_device_batch'           : BATCH_SIZE,
    'weight_decay'               : 0.01,
    'max_grad_norm'              : 1.0,
    'fp16'                       : FP16,
    'gradient_accumulation_steps': GRAD_ACCUM,
    'early_stopping_patience'    : 3,
    'train_val_split'            : 0.93,
    'warmup_ratio'               : 0.06,
    'seed'                       : SEED,
    'device'                     : DEVICE,
    'use_gpu'                    : USE_GPU,
    # [BUG FIX #1] do_sample=False → PAS de temperature
    'gen_extract': {'do_sample': False, 'max_new_tokens': 120},
    'gen_chat':   {'do_sample': True,  'temperature': 0.75, 'top_p': 0.92,
                   'top_k': 50, 'repetition_penalty': 1.30, 'max_new_tokens': 80},
    'gen_eval':   {'do_sample': True,  'temperature': 0.30, 'top_p': 0.85,
                   'top_k': 30, 'repetition_penalty': 1.40, 'max_new_tokens': 80},
}

print('='*65)
print(f"  Dispositif : {DEVICE.upper()} {'✅ GPU' if USE_GPU else '⚠️ CPU'}")
if USE_GPU:
    print(f"  GPU   : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  Batch : {BATCH_SIZE} | Grad accum : {GRAD_ACCUM} | FP16 : {FP16}")
print(f"  CSV   : {CSV_PATH}")
print('='*65)

  Dispositif : CUDA ✅ GPU
  GPU   : Tesla T4
  VRAM  : 15.6 GB
  Batch : 8 | Grad accum : 1 | FP16 : True
  CSV   : /kaggle/input/datasets/jmaighofran/data-immobiler-finale-prete-csv/daaataaa.csv


In [4]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 4 — Chargement & nettoyage du dataset
# ════════════════════════════════════════════════════════════════
import unicodedata as _ud
print('[ÉTAPE 4] Chargement...')

df = pd.read_csv(CONFIG['csv_path'])
print(f"  ✓ CSV brut : {len(df):,} lignes × {len(df.columns)} colonnes")

PTYPE_MAP = {'villa':'villa','appart':'appartement','appartement':'appartement',
             'studio':'studio','maison':'villa','local':'local'}

def _norm_ptype(raw):
    s = str(raw).lower().strip()
    for k, v in PTYPE_MAP.items():
        if k in s: return v
    return s

def _norm_str(s):
    s = str(s).lower().strip()
    return ''.join(c for c in _ud.normalize('NFD', s) if _ud.category(c) != 'Mn')

if 'property_type' in df.columns:
    df['property_type'] = df['property_type'].apply(_norm_ptype)

df['bedrooms']       = pd.to_numeric(df.get('bedrooms', pd.Series(dtype=float)), errors='coerce').fillna(2.0)
df['price_numeric']  = pd.to_numeric(df.get('price_numeric',  pd.Series(dtype=float)), errors='coerce')
df['area_m2_numeric']= pd.to_numeric(df.get('area_m2_numeric',pd.Series(dtype=float)), errors='coerce')

for col in ['piscine','jardin','garage','meuble','climatisation','ascenseur']:
    if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    else: df[col] = 0

initial = len(df)
df = df.dropna(subset=['price_numeric','area_m2_numeric'])
df = df[(df['price_numeric'] > 50) & (df['price_numeric'] < 10_000_000)]
df = df[(df['area_m2_numeric'] > 20) & (df['area_m2_numeric'] < 2000)]

df['ville_lower']         = df['ville'].astype(str).str.lower().str.strip()
df['ville_norm']          = df['ville'].apply(_norm_str)
df['property_type_lower'] = df['property_type'].astype(str).str.lower().str.strip()
df = df.reset_index(drop=True)

print(f"  ✓ Après nettoyage : {len(df):,} biens (sur {initial:,})")
print(f"  ✓ Types : {df['property_type'].value_counts().to_dict()}")
print(f"  ✓ Vente : {len(df[df['type']=='vente']):,} | Location : {len(df[df['type']=='location']):,}")

[ÉTAPE 4] Chargement...
  ✓ CSV brut : 30,023 lignes × 20 colonnes
  ✓ Après nettoyage : 29,723 biens (sur 30,023)
  ✓ Types : {'appartement': 21785, 'villa': 7938}
  ✓ Vente : 11,755 | Location : 17,968


In [5]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 5 — Prompt Engineering (paires d'entraînement)
# ════════════════════════════════════════════════════════════════
print('[ÉTAPE 5] Génération des paires...')

FEW_SHOT_EXTRACTION = [
    {'msg': 'Je veux une villa S+3 à La Marsa avec piscine, budget max 500K DT',
     'json': '{"villes":["La Marsa"],"property_type":"villa","piscine":1,"jardin":0,"garage":0,"meuble":0,"budget_min":null,"budget_max":500000,"bedrooms_exact":3}'},
    {'msg': 'Appartement S+2 à Tunis max 300000 DT meublé avec garage',
     'json': '{"villes":["Tunis"],"property_type":"appartement","piscine":0,"jardin":0,"garage":1,"meuble":1,"budget_min":null,"budget_max":300000,"bedrooms_exact":2}'},
    {'msg': 'Villa entre 300K et 900K DT à Gammarth ou La Marsa avec jardin',
     'json': '{"villes":["Gammarth","La Marsa"],"property_type":"villa","piscine":0,"jardin":1,"garage":0,"meuble":0,"budget_min":300000,"budget_max":900000,"bedrooms_exact":null}'},
    {'msg': 'Louer un S+2 à Sousse avec piscine et parking',
     'json': '{"villes":["Sousse"],"property_type":"appartement","piscine":1,"jardin":0,"garage":1,"meuble":0,"budget_min":null,"budget_max":null,"bedrooms_exact":2}'},
    {'msg': 'Studio meublé à Ariana budget 150K DT maximum',
     'json': '{"villes":["Ariana"],"property_type":"studio","piscine":0,"jardin":0,"garage":0,"meuble":1,"budget_min":null,"budget_max":150000,"bedrooms_exact":1}'},
    {'msg': 'Villa S+4 piscine jardin garage à Carthage, budget max 2 millions',
     'json': '{"villes":["Carthage"],"property_type":"villa","piscine":1,"jardin":1,"garage":1,"meuble":0,"budget_min":null,"budget_max":2000000,"bedrooms_exact":4}'},
    {'msg': 'Cherche appartement S+3 meublé à El Menzah',
     'json': '{"villes":["El Menzah"],"property_type":"appartement","piscine":0,"jardin":0,"garage":0,"meuble":1,"budget_min":null,"budget_max":null,"bedrooms_exact":3}'},
    {'msg': 'S+2 à louer à Bizerte avec jardin, max 1500 DT par mois',
     'json': '{"villes":["Bizerte"],"property_type":"appartement","piscine":0,"jardin":1,"garage":0,"meuble":0,"budget_min":null,"budget_max":1500,"bedrooms_exact":2}'},
]

EVAL_EXAMPLES = [
    {'desc': 'Villa architecte Carthage piscine débordement jardin 1200m² garage double marbre domotique',
     'dim': 'Prestige & Luxe',
     'rep': "SCORE: 9/10 | REASON: Villa d'architecte avec piscine à débordement et domotique dans quartier prestige."},
    {'desc': 'Appartement S+2 3ème étage sans ascenseur cuisine basique pas de parking',
     'dim': 'Prestige & Luxe',
     'rep': 'SCORE: 2/10 | REASON: Appartement standard sans équipements premium.'},
    {'desc': 'Villa S+4 La Marsa piscine privée jardin 600m² garage finitions haut de gamme',
     'dim': 'Équipements Premium',
     'rep': 'SCORE: 8/10 | REASON: Piscine privée, jardin spacieux et garage — équipements premium.'},
    {'desc': 'Studio 35m² Ariana proche métro idéal investissement locatif',
     'dim': 'Surface & Architecture',
     'rep': 'SCORE: 2/10 | REASON: Petite surface 35m², aucune mention architecture.'},
    {'desc': 'Villa luxueuse Sidi Bou Said vue mer piscine chauffée jardin domotique 5 chambres',
     'dim': 'Localisation & Quartier',
     'rep': 'SCORE: 10/10 | REASON: Sidi Bou Said est le quartier le plus prestigieux de Tunisie.'},
    {'desc': 'Villa moderne La Marsa 2021 320m² terrain 500m² piscine jardin garage cuisine équipée',
     'dim': 'Qualité de la Description',
     'rep': 'SCORE: 9/10 | REASON: Description très détaillée avec surface, année, équipements.'},
]

pairs = []

# Type A : Extraction JSON
for ex in FEW_SHOT_EXTRACTION:
    inp = (f'[INSTRUCTION: Tu es AQARIA. Extrais UNIQUEMENT un JSON valide '
           f'avec les critères immobiliers.]\nMessage: {ex["msg"]}\nJSON:')
    pairs.append({'input': inp, 'target': ' ' + ex['json']})
    inp2 = (f'[INSTRUCTION: AQARIA extraction critères JSON immobilier Tunisie.]\n'
            f'Demande: {ex["msg"]}\nCritères JSON:')
    pairs.append({'input': inp2, 'target': ' ' + ex['json']})

# Type B : Prix par ville
cities = df['ville'].dropna().unique()[:25]
for ville in cities:
    for ptype in ['villa', 'appartement']:
        sub = df[(df['ville'] == ville) & (df['property_type_lower'] == ptype)]
        if len(sub) < 3: continue
        p10 = int(sub['price_numeric'].quantile(0.10))
        p90 = int(sub['price_numeric'].quantile(0.90))
        moy = int(sub['price_numeric'].mean())
        inp = (f'[INSTRUCTION: Tu es AQARIA, expert immobilier tunisien.]\n'
               f'Question: Quel est le prix d\'un {ptype} à {ville} ?\nRéponse:')
        tgt = (f' À {ville}, un {ptype} coûte entre {p10:,} et {p90:,} DT '
               f'(moyenne {moy:,} DT). {len(sub)} annonces disponibles.')
        pairs.append({'input': inp, 'target': tgt})

# Type C : Recommandations de biens réels
sample_rows = df.sample(min(600, len(df)), random_state=SEED)
for _, row in sample_rows.iterrows():
    ville  = str(row.get('ville', 'Inconnue'))
    ptype  = str(row.get('property_type', 'bien')).lower()
    price  = int(row['price_numeric'])
    area   = int(row['area_m2_numeric'])
    beds   = int(row.get('bedrooms', 0) or 0)
    title  = str(row.get('title', '')).replace('\n', ' ').strip()[:80]
    ams    = [f for f in ['piscine','jardin','garage','meuble'] if row.get(f, 0) == 1]
    am_str = (', avec ' + ', '.join(ams)) if ams else ''
    splus  = max(0, beds - 1)
    inp = (f'[INSTRUCTION: Tu es AQARIA, recommande des biens immobiliers en Tunisie.]\n'
           f'Question: Trouve-moi un {ptype} S+{splus} à {ville}{am_str} '
           f'budget {price:,} DT\nRéponse:')
    tgt = (f' J\'ai trouvé : {title}. Surface {area}m², S+{splus}{am_str}. '
           f'Prix : {price:,} DT. Situé à {ville}.')
    pairs.append({'input': inp, 'target': tgt})

# Type D : Dialogues généraux
DIALOGUES = [
    ('Bonjour, que peux-tu faire ?',
     ' Je suis AQARIA, votre assistant immobilier tunisien. Je trouve des villas, '
     'appartements et studios selon votre budget et vos critères.'),
    ('Quelles villes recommandes-tu pour investir ?',
     ' La Marsa et Carthage offrent la plus forte valorisation. Ariana et El Menzah '
     'sont équilibrés. Sousse et Hammamet sont attractifs pour le locatif.'),
    ('Quelle est la différence entre vente et location ?',
     ' La vente est un achat définitif avec transfert de propriété. La location est '
     'un loyer mensuel sans acquisition. Les prix de vente sont en DT, les loyers en DT/mois.'),
    ('C\'est quoi S+2 ?',
     ' S+2 désigne un salon + 2 chambres (3 pièces au total). S+3 = salon + 3 chambres.'),
    ('Quel est le marché immobilier à Tunis ?',
     ' Le marché tunisois est dynamique. Les appartements S+2 varient entre 150K et 400K DT. '
     'Les villas à Carthage dépassent souvent 1M DT.'),
]
for q, r in DIALOGUES:
    inp = (f'[INSTRUCTION: Tu es AQARIA, assistant immobilier tunisien expert.]\n'
           f'Question: {q}\nRéponse:')
    pairs.append({'input': inp, 'target': r})

# Type E : Évaluation B2B
for ex in EVAL_EXAMPLES:
    inp = (f'[INSTRUCTION: Tu es un expert évaluateur immobilier tunisien B2B. '
           f'Évalue sur la dimension "{ex["dim"]}". Format: SCORE: X/10 | REASON: phrase.]\n'
           f'Annonce: {ex["desc"]}\nÉvaluation:')
    pairs.append({'input': inp, 'target': ' ' + ex['rep']})

random.shuffle(pairs)
print(f"  ✓ Total paires : {len(pairs)} (JSON:{len(FEW_SHOT_EXTRACTION)*2}, Recomm:~600, Dial:{len(DIALOGUES)}, Eval:{len(EVAL_EXAMPLES)})")

[ÉTAPE 5] Génération des paires...
  ✓ Total paires : 653 (JSON:16, Recomm:~600, Dial:5, Eval:6)


In [6]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 6 — Chargement GPT-2 + Tokenizer
# ════════════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer
print('[ÉTAPE 6] Chargement GPT-2...')

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'  # [BUG FIX #4]

model = AutoModelForCausalLM.from_pretrained('gpt2').to(CONFIG['device'])

n_params = sum(p.numel() for p in model.parameters())
print(f"  ✓ GPT-2 chargé sur {CONFIG['device'].upper()}")
print(f"  ✓ Paramètres : {n_params:,} (~124M)")
print(f"  ✓ padding_side : '{tokenizer.padding_side}' [CORRIGÉ]")
if USE_GPU:
    print(f"  ✓ VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB")

[ÉTAPE 6] Chargement GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ GPT-2 chargé sur CUDA
  ✓ Paramètres : 124,439,808 (~124M)
  ✓ padding_side : 'right' [CORRIGÉ]
  ✓ VRAM : 0.51 GB


In [7]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 7 — Dataset PyTorch avec Label Masking
# [BUG FIX #3] collate_fn avec padding dynamique
# ════════════════════════════════════════════════════════════════
from torch.utils.data import Dataset, DataLoader

class AqariaDataset(Dataset):
    def __init__(self, data, tok, max_len=256):
        self.data, self.tok, self.max_len = data, tok, max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item      = self.data[idx]
        full_text = item['input'].strip() + ' ' + item['target'].strip() + self.tok.eos_token
        enc       = self.tok(full_text, max_length=self.max_len, truncation=True, return_tensors='pt')
        input_ids = enc['input_ids'].squeeze(0)
        prompt_enc= self.tok(item['input'].strip(), max_length=self.max_len, truncation=True, return_tensors='pt')
        prompt_len= prompt_enc['input_ids'].shape[1]
        labels    = input_ids.clone()
        labels[:prompt_len] = -100  # Masquer le prompt — loss=0 ici
        return {'input_ids': input_ids, 'labels': labels}

def collate_batch(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id  = tokenizer.pad_token_id
    input_ids_b, attn_b, labels_b = [], [], []
    for item in batch:
        seq = item['input_ids'].shape[0]
        pad = max_len - seq
        input_ids_b.append(torch.cat([item['input_ids'], torch.full((pad,), pad_id, dtype=torch.long)]))
        attn_b.append(torch.cat([torch.ones(seq, dtype=torch.long), torch.zeros(pad, dtype=torch.long)]))
        labels_b.append(torch.cat([item['labels'], torch.full((pad,), -100, dtype=torch.long)]))
    return {'input_ids': torch.stack(input_ids_b),
            'attention_mask': torch.stack(attn_b),
            'labels': torch.stack(labels_b)}

split_idx  = int(len(pairs) * CONFIG['train_val_split'])
train_data = pairs[:split_idx]
val_data   = pairs[split_idx:]

train_ds = AqariaDataset(train_data, tokenizer, CONFIG['max_seq_length'])
val_ds   = AqariaDataset(val_data,   tokenizer, CONFIG['max_seq_length'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['per_device_batch'],
                          shuffle=True,  collate_fn=collate_batch, num_workers=0, pin_memory=USE_GPU)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['per_device_batch'],
                          shuffle=False, collate_fn=collate_batch, num_workers=0, pin_memory=USE_GPU)

print(f"  ✓ Train : {len(train_ds)} exemples, {len(train_loader)} batches")
print(f"  ✓ Val   : {len(val_ds)} exemples, {len(val_loader)} batches")
print(f"  ✓ collate_fn : padding dynamique [CORRIGÉ]")

  ✓ Train : 607 exemples, 76 batches
  ✓ Val   : 46 exemples, 6 batches
  ✓ collate_fn : padding dynamique [CORRIGÉ]


In [8]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 8 — Fine-tuning GPT-2
# [BUG FIX #5] Validation complète + Early Stopping
# [BUG FIX #6] Import correct get_cosine_schedule_with_warmup
# [BUG FIX #9] warmup_steps calculé après DataLoader
# ════════════════════════════════════════════════════════════════
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Optimizer avec weight decay découplé
no_decay = ['bias', 'LayerNorm.weight']
optimizer_groups = [
    {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': CONFIG['weight_decay']},
    {'params': [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = AdamW(optimizer_groups, lr=CONFIG['learning_rate'])

# [BUG FIX #9] Calcul après création du DataLoader
total_steps  = len(train_loader) * CONFIG['num_train_epochs'] // CONFIG['gradient_accumulation_steps']
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler    = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = torch.cuda.amp.GradScaler() if CONFIG['fp16'] and USE_GPU else None

print(f"  ✓ Optimizer : AdamW | LR={CONFIG['learning_rate']} | WD={CONFIG['weight_decay']}")
print(f"  ✓ Steps : {total_steps} total, {warmup_steps} warmup")
print(f"  ✓ FP16 : {'Activé' if scaler else 'Désactivé'}")
print()
print('Lancement du fine-tuning...')
print('='*65)

best_val_loss   = float('inf')
patience_count  = 0
best_model_path = os.path.join(CONFIG['output_dir'], 'best_model')
train_start     = time.time()

for epoch in range(CONFIG['num_train_epochs']):
    # ── ENTRAÎNEMENT ──
    model.train()
    train_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(CONFIG['device']) for k,v in batch.items()}
        if scaler:
            with torch.cuda.amp.autocast():
                outputs = model(**batch)
                loss    = outputs.loss / CONFIG['gradient_accumulation_steps']
            scaler.scale(loss).backward()
        else:
            outputs = model(**batch)
            loss    = outputs.loss / CONFIG['gradient_accumulation_steps']
            loss.backward()
        train_loss += outputs.loss.item()
        if (step + 1) % CONFIG['gradient_accumulation_steps'] == 0:
            if scaler:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                scaler.step(optimizer); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    avg_train = train_loss / len(train_loader)

    # ── VALIDATION [BUG FIX #5] ──
    model.eval()  # [BUG FIX #2]
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(CONFIG['device']) for k,v in batch.items()}
            val_loss += model(**batch).loss.item()
    avg_val = val_loss / len(val_loader)

    elapsed = (time.time() - train_start) / 60
    print(f"  Epoch {epoch+1}/{CONFIG['num_train_epochs']} | "
          f"Train: {avg_train:.4f} | Val: {avg_val:.4f} | "
          f"LR: {scheduler.get_last_lr()[0]:.2e} | {elapsed:.1f}min")

    if avg_val < best_val_loss:
        best_val_loss  = avg_val
        patience_count = 0
        model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        print(f"    ✅ Meilleur modèle sauvegardé (val={best_val_loss:.4f})")
    else:
        patience_count += 1
        print(f"    ⚠️ Patience {patience_count}/{CONFIG['early_stopping_patience']}")
        if patience_count >= CONFIG['early_stopping_patience']:
            print(f"  🛑 Early stopping à l'epoch {epoch+1}")
            break

print('='*65)
print(f"  ✅ Fine-tuning terminé en {(time.time()-train_start)/60:.1f} min")
print(f"  ✅ Meilleur val_loss : {best_val_loss:.4f}")

  ✓ Optimizer : AdamW | LR=0.0003 | WD=0.01
  ✓ Steps : 380 total, 22 warmup
  ✓ FP16 : Activé

Lancement du fine-tuning...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  Epoch 1/5 | Train: 1.1153 | Val: 0.7426 | LR: 2.83e-04 | 0.2min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Meilleur modèle sauvegardé (val=0.7426)
  Epoch 2/5 | Train: 0.4222 | Val: 0.6792 | LR: 2.13e-04 | 0.4min


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✅ Meilleur modèle sauvegardé (val=0.6792)
  Epoch 3/5 | Train: 0.3189 | Val: 0.6913 | LR: 1.15e-04 | 0.6min
    ⚠️ Patience 1/3
  Epoch 4/5 | Train: 0.2515 | Val: 0.6949 | LR: 3.21e-05 | 0.8min
    ⚠️ Patience 2/3
  Epoch 5/5 | Train: 0.2056 | Val: 0.7342 | LR: 0.00e+00 | 1.0min
    ⚠️ Patience 3/3
  🛑 Early stopping à l'epoch 5
  ✅ Fine-tuning terminé en 1.0 min
  ✅ Meilleur val_loss : 0.6792


In [9]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 9 — Rechargement du meilleur modèle + eval()
# [BUG FIX #2] model.eval() explicite
# ════════════════════════════════════════════════════════════════
from transformers import AutoModelForCausalLM, AutoTokenizer
print('[ÉTAPE 9] Rechargement du meilleur modèle...')

if os.path.exists(best_model_path):
    model     = AutoModelForCausalLM.from_pretrained(best_model_path).to(CONFIG['device'])
    tokenizer = AutoTokenizer.from_pretrained(best_model_path)
    tokenizer.padding_side = 'right'
    print(f"  ✓ Modèle rechargé depuis : {best_model_path}")
else:
    print("  ⚠️ Pas de modèle sauvegardé — utilisation du modèle courant")

model.eval()  # [BUG FIX #2] OBLIGATOIRE
print("  ✓ model.eval() [CORRIGÉ]")

_inference_lock = threading.Lock()  # [BUG FIX #7]
print("  ✓ threading.Lock() créé [CORRIGÉ]")

if USE_GPU:
    print(f"  ✓ VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB")

[ÉTAPE 9] Rechargement du meilleur modèle...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  ✓ Modèle rechargé depuis : ./models_aqaria_gpt2_finetuned/best_model
  ✓ model.eval() [CORRIGÉ]
  ✓ threading.Lock() créé [CORRIGÉ]
  ✓ VRAM : 2.17 GB


In [10]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 10 — Fonctions Chat & Évaluation
# Détection automatique du modèle selon l'intention
# ════════════════════════════════════════════════════════════════

def _generate_safe(prompt: str, gen_params: dict) -> str:
    """Génération thread-safe. [BUG FIX #1, #2, #7]"""
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=CONFIG['max_seq_length'])
    inputs = {k: v.to(CONFIG['device']) for k,v in inputs.items()}
    with _inference_lock:
        with torch.no_grad():
            output = model.generate(**inputs,
                                    pad_token_id=tokenizer.eos_token_id,
                                    **gen_params)
    return tokenizer.decode(output[0], skip_special_tokens=True)


def detect_model(message: str) -> str:
    """
    Détection automatique du modèle à utiliser selon l'intention.
    Retourne : 'extract' | 'chat' | 'eval'
    """
    t = message.lower()
    # Normalisation accents
    t = ''.join(c for c in _ud.normalize('NFD', t) if _ud.category(c) != 'Mn')
    if re.search(r'evalue|score|analyse|audit|noter|premium|evaluer|notez', t):
        return 'eval'
    if re.search(r'villa|appart|studio|piscine|jardin|garage|meuble|cherche|trouve|louer|location|s\+\d|budget', t):
        return 'extract'
    return 'chat'


def chat_aqaria(message: str) -> str:
    """Chatbot AQARIA — détection auto + filtrage CSV + réponse GPT-2."""
    mode = detect_model(message)

    # ── Mode EXTRACTION JSON (RAG) ──
    if mode == 'extract':
        prompt_e = (f'[INSTRUCTION: Tu es AQARIA. Extrais UNIQUEMENT un JSON valide '
                    f'avec les critères immobiliers.]\nMessage: {message}\nJSON:')
        raw = _generate_safe(prompt_e, CONFIG['gen_extract'])  # greedy
        raw = raw.split('JSON:')[-1].strip() if 'JSON:' in raw else raw

        criteres = {'villes': [], 'property_type': None, 'piscine': 0,
                    'jardin': 0, 'garage': 0, 'meuble': 0,
                    'budget_min': None, 'budget_max': None, 'bedrooms_exact': None}
        try:
            m = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
            if m: criteres = json.loads(m.group(0))
        except Exception: pass

        # Fallback regex
        tl = message.lower()
        tl_norm = ''.join(c for c in _ud.normalize('NFD', tl) if _ud.category(c) != 'Mn')
        KNC = sorted([str(v).lower().strip() for v in df['ville'].dropna().unique()], key=lambda x: -len(x))
        if not criteres.get('villes'):
            for city in KNC:
                if city and city in tl: criteres['villes'] = [city.title()]; break
        if not criteres.get('property_type'):
            if re.search(r'\bvilla\b', tl): criteres['property_type'] = 'villa'
            elif re.search(r'\b(appartement|appart)\b', tl): criteres['property_type'] = 'appartement'
            elif re.search(r'\bstudio\b', tl): criteres['property_type'] = 'studio'
        for eq in ['piscine', 'jardin', 'garage']:
            if eq in tl and not criteres.get(eq): criteres[eq] = 1
        if re.search(r'meub', tl_norm) and not criteres.get('meuble'): criteres['meuble'] = 1
        if re.search(r'\b(louer|location|locat)\b', tl) and not criteres.get('type_transaction'):
            criteres['type_transaction'] = 'location'
        if not criteres.get('bedrooms_exact'):
            m2 = re.search(r's\+(\d)', tl)
            if m2: criteres['bedrooms_exact'] = int(m2.group(1))
        nums = [int(re.sub(r'\D','',b)) for b in re.findall(r'(\d[\d\s]*(?:000|k|m|dt))', tl, re.IGNORECASE) if re.sub(r'\D','',b)]
        if nums and not criteres.get('budget_max'): criteres['budget_max'] = max(nums)

        biens_df, note = rechercher_biens(criteres, n=6)
        nb = len(biens_df)
        if nb > 0:
            ville_str = criteres['villes'][0] if criteres.get('villes') else 'Tunisie'
            ex = biens_df.iloc[0]
            ttl = str(ex.get('title',''))[:60]
            pr  = int(ex['price_numeric'])
            reply = f"Trouvé {nb} bien(s) à {ville_str}. {ttl}, {pr:,} DT."
            # Générer suite conversationnelle
            prompt_c = (f'[INSTRUCTION: Tu es AQARIA, assistant immobilier tunisien.]\n'
                        f'Question: {message}\nRéponse:')
            suite = _generate_safe(prompt_c, CONFIG['gen_chat'])
            suite = suite.split('Réponse:')[-1].strip() if 'Réponse:' in suite else suite
            reply = (reply + ' ' + suite)[:350]
        else:
            reply = note or 'Aucun bien trouvé avec ces critères. Essayez avec moins de filtres.'
        return reply, biens_df, criteres, mode

    # ── Mode CHAT ──
    elif mode == 'chat':
        prompt_c = (f'[INSTRUCTION: Tu es AQARIA, expert immobilier tunisien. '
                    f'Réponds en français de manière utile et concise.]\n'
                    f'Question: {message}\nRéponse:')
        reply = _generate_safe(prompt_c, CONFIG['gen_chat'])
        reply = reply.split('Réponse:')[-1].strip() if 'Réponse:' in reply else reply
        return reply[:400], None, {}, mode

    # ── Mode EVAL ──
    else:
        res = evaluer_annonce(message, verbose=False)
        reply = (f"Score : {res['score']}/100 — {res['categorie']}. "
                 f"{res.get('recommandation','')}")
        return reply[:400], None, {}, mode


print('  ✓ detect_model() défini — détection automatique chat/RAG/eval')
print('  ✓ chat_aqaria() défini — retourne (reply, biens_df, criteres, mode)')

  ✓ detect_model() défini — détection automatique chat/RAG/eval
  ✓ chat_aqaria() défini — retourne (reply, biens_df, criteres, mode)


In [11]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 10b — Évaluateur Premium B2B (5 dimensions)
# ════════════════════════════════════════════════════════════════

def _gen_recommandation(score: float) -> str:
    if score >= 85: return '✅ Annonce prête pour publication haut de gamme.'
    if score >= 70: return '✅ Bonne annonce. Ajoutez des photos professionnelles.'
    if score >= 50: return '⚠️ Enrichissez la description avec équipements et quartier exact.'
    return '❌ Description insuffisante. Développez les atouts et la localisation.'


def evaluer_annonce(texte: str, verbose: bool = False) -> dict:
    """Évalue une annonce sur 5 dimensions, retourne dict JSON-serializable."""
    DIMS = {
        'prestige':     ('Prestige & Luxe',           0.30),
        'equipements':  ('Équipements Premium',        0.25),
        'surface':      ('Surface & Architecture',     0.20),
        'localisation': ('Localisation & Quartier',    0.15),
        'description':  ('Qualité de la Description',  0.10),
    }
    resultats, score_global = {}, 0.0

    for dim_key, (dim_label, poids) in DIMS.items():
        prompt = (f'[INSTRUCTION: Tu es un expert évaluateur immobilier tunisien B2B. '
                  f'Évalue sur la dimension "{dim_label}". Format: SCORE: X/10 | REASON: phrase.]\n'
                  f'Annonce: {texte[:300]}\nÉvaluation:')
        rep = _generate_safe(prompt, CONFIG['gen_eval'])
        rep = rep.split('Évaluation:')[-1].strip() if 'Évaluation:' in rep else rep

        score = 5.0
        m = re.search(r'SCORE:\s*(\d+(?:\.\d+)?)', rep, re.IGNORECASE)
        if m: score = min(10.0, float(m.group(1)))
        else:
            m = re.search(r'(\d+(?:\.\d+)?)\s*/\s*10', rep)
            if m: score = min(10.0, float(m.group(1)))

        justif = ''
        m2 = re.search(r'REASON:\s*(.+)', rep, re.IGNORECASE)
        if m2: justif = m2.group(1).strip()[:120]

        resultats[dim_key] = {'label': dim_label, 'score': float(score),
                              'poids': poids, 'justif': justif}
        score_global += score * poids
        if verbose: print(f"  {dim_label:<35} → {score:4.1f}/10  (×{poids:.0%})")

    sf = round(score_global * 10, 1)
    if   sf >= 85: cat = '🏆 EXCEPTIONNEL — Prestige Supérieur'
    elif sf >= 70: cat = '⭐ HAUT STANDING'
    elif sf >= 50: cat = '🏠 STANDARD — Bon rapport qualité/prix'
    elif sf >= 30: cat = '🔧 ENTRÉE DE GAMME'
    else:          cat = '🔨 POPULAIRE / À RÉNOVER'

    if verbose:
        print(f"  SCORE GLOBAL : {sf}/100")
        print(f"  CATÉGORIE   : {cat}")

    return {'score': sf, 'categorie': cat,
            'dimensions': resultats,
            'recommandation': _gen_recommandation(sf)}


# ── Test rapide ──
print('🧪 Test évaluation rapide...')
r = evaluer_annonce('Villa architecte Carthage piscine débordement marbre Carrare domotique jardin 1200m².', verbose=False)
print(f"  Score : {r['score']}/100 — {r['categorie']}")

🧪 Test évaluation rapide...
  Score : 50.0/100 — 🏠 STANDARD — Bon rapport qualité/prix


In [12]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 11 — Moteur de filtrage CSV
# ════════════════════════════════════════════════════════════════
import unicodedata as _ud2

KNOWN_CITIES    = sorted([str(v).lower().strip() for v in df['ville'].dropna().unique()], key=lambda x: -len(x))
SPLUS_TO_BEDS   = {1: 2.0, 2: 3.0, 3: 4.0}
BEDS_TO_SPLUS   = {2.0: 1, 3.0: 2, 4.0: 3}

def _norm_q(s):
    s = str(s).lower().strip()
    return ''.join(c for c in _ud2.normalize('NFD', s) if _ud2.category(c) != 'Mn')

def rechercher_biens(criteres: dict, n: int = 6):
    res, note = df.copy(), ''

    # Filtre ville
    villes = criteres.get('villes', [])
    if villes:
        vl_norm = [_norm_q(v) for v in villes]
        res = res[res['ville_norm'].apply(lambda x: any(v in x or x in v for v in vl_norm))]
        if len(res) == 0:
            res = df.copy()
            res = res[res['ville_norm'].apply(lambda x: any(v[:4] in x for v in vl_norm if len(v) >= 4))]
            if len(res) > 0: note = '⚠️ Correspondance approchée pour la ville.'

    # Filtre property_type
    ptype = criteres.get('property_type')
    if ptype:
        _v = {'villa': ['villa'], 'appartement': ['appartement'], 'studio': ['appartement','studio']}
        variants = _v.get(ptype.lower(), [ptype.lower()])
        res = res[res['property_type_lower'].apply(lambda x: any(v in x for v in variants))]

    # Filtre type transaction
    trans = criteres.get('type_transaction')
    if trans and 'type' in res.columns:
        _t = 'location' if 'locat' in str(trans).lower() or 'louer' in str(trans).lower() else 'vente'
        res = res[res['type'] == _t]

    # Filtre équipements
    for eq in ['piscine', 'jardin', 'garage', 'meuble']:
        if criteres.get(eq) == 1 and eq in res.columns:
            res = res[res[eq] == 1]

    # Filtre budget
    bmax = criteres.get('budget_max')
    bmin = criteres.get('budget_min')
    if bmax: res = res[res['price_numeric'] <= bmax]
    if bmin: res = res[res['price_numeric'] >= bmin]

    # Filtre bedrooms S+X
    beds_exact = criteres.get('bedrooms_exact')
    if beds_exact is not None:
        target_splus = int(beds_exact)
        target_beds  = SPLUS_TO_BEDS.get(target_splus)
        if target_beds is None: target_beds = 4.0; note = f'⚠️ S+{target_splus} non disponible. Voici S+3 :'
        res_exact = res[res['bedrooms'] == target_beds]
        if len(res_exact) > 0:
            res = res_exact
        else:
            for alt in [target_beds - 1, target_beds + 1]:
                if alt in BEDS_TO_SPLUS.values():
                    res_alt = res[res['bedrooms'] == alt]
                    if len(res_alt) > 0:
                        note = f'⚠️ Aucun S+{target_splus} exact. Voici S+{BEDS_TO_SPLUS.get(alt,"?")} :'
                        res  = res_alt; break

    res = res.drop_duplicates(subset=['title']).sort_values('price_numeric')
    return res.head(n), note


# ── Tests ──
t1, _ = rechercher_biens({'villes': ['La Marsa'], 'property_type': 'villa', 'piscine': 1}, n=3)
print(f"  ✓ Villa+piscine La Marsa : {len(t1)} résultats")
t2, _ = rechercher_biens({'villes': ['Ariana'], 'property_type': 'appartement',
                           'type_transaction': 'location', 'bedrooms_exact': 2}, n=3)
print(f"  ✓ Appart S+2 Ariana location : {len(t2)} résultats")

  ✓ Villa+piscine La Marsa : 3 résultats
  ✓ Appart S+2 Ariana location : 3 résultats


In [13]:
# ════════════════════════════════════════════════════════════════
# FLASK COMPLET — Initialisation + Routes + Démarrage
# À exécuter comme une seule cellule
# ════════════════════════════════════════════════════════════════

import threading, time, urllib.request, psutil
from flask import Flask, request, jsonify
from flask_cors import CORS
import json as _json, re as _re

# ── 1. Libération du port ──
FLASK_PORT = 5000

def _kill_port(port):
    for proc in psutil.process_iter(['pid', 'connections']):
        try:
            for conn in (proc.info.get('connections') or []):
                if hasattr(conn, 'laddr') and conn.laddr.port == port:
                    proc.kill()
                    print(f"  ✓ PID {proc.info['pid']} tué (port {port})")
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass

_kill_port(FLASK_PORT)

# ── 2. Création de l'application ──
app_flask = Flask(__name__)
CORS(app_flask, resources={r"/api/*": {"origins": "*"}})

# Variables globales (déjà définies dans les cellules précédentes)
_model     = model
_tokenizer = tokenizer
_CONFIG    = CONFIG
_df        = df
_lock      = _inference_lock

print(f"  ✅ Flask app créée — port {FLASK_PORT}")

# ── 3. TOUTES LES ROUTES (AVANT démarrage) ──

@app_flask.route('/api/status', methods=['GET'])
def api_status():
    return jsonify({
        'status': 'ok',
        'model': 'GPT-2 fine-tuné AQARIA',
        'biens': len(_df),
        'villes': list(_df['ville'].dropna().unique()[:8]),
    })

@app_flask.route('/api/recommander', methods=['POST', 'OPTIONS'])
def api_recommander():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json(force=True, silent=True) or {}
    message = data.get('message', data.get('query', '')).strip()
    n = int(data.get('n', 6))
    if not message:
        return jsonify({'response': {'text': 'Message vide.', 'biens': []}}), 400
    
    from chat_functions import detect_model, extract_criteres, rechercher_biens
    from config import BEDS_TO_SPLUS
    
    mode = detect_model(message)
    
    if mode == 'extract':
        criteres = extract_criteres(message)
        biens_df, _ = rechercher_biens(criteres, n=n)
        biens_list = []
        for _, row in biens_df.iterrows():
            biens_list.append({
                'title': str(row.get('title', ''))[:80],
                'price': int(row['price_numeric']),
                'area': int(row['area_m2_numeric']),
                'bedrooms': BEDS_TO_SPLUS.get(row['bedrooms'], '?'),
                'ville': str(row.get('ville', '')),
            })
        if biens_list:
            reply = f"Trouvé {len(biens_list)} bien(s)."
        else:
            reply = "Aucun bien trouvé."
        return jsonify({'response': {'text': reply, 'biens': biens_list}})
    else:
        # Mode chat
        from chat_functions import generate_chat_response
        reply = generate_chat_response(message)
        return jsonify({'response': {'text': reply, 'biens': []}})

@app_flask.route('/api/evaluer', methods=['POST', 'OPTIONS'])
def api_evaluer():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json(force=True, silent=True) or {}
    desc = data.get('description', data.get('texte', '')).strip()
    if not desc:
        return jsonify({'error': 'description manquante'}), 400
    from chat_functions import evaluer_annonce
    result = evaluer_annonce(desc, verbose=False)
    return jsonify(result)

@app_flask.route('/api/chat', methods=['POST', 'OPTIONS'])
def api_chat():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    return api_recommander()

@app_flask.route('/api/rag', methods=['POST', 'OPTIONS'])
def api_rag():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json(force=True, silent=True) or {}
    query = data.get('q', data.get('message', '')).strip()
    if not query:
        return jsonify({'results': []}), 400
    from search_engine import search_similar
    results = search_similar(query, n=6)
    return jsonify({'results': results, 'total': len(results)})

print("  ✅ Routes enregistrées")

# ── 4. Démarrage du serveur (APRÈS les routes) ──
def _run_flask():
    app_flask.run(host='0.0.0.0', port=FLASK_PORT,
                  debug=False, use_reloader=False, threaded=True)

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()

print(f"  ⏳ Démarrage Flask sur le port {FLASK_PORT}...", end='', flush=True)
for _i in range(30):
    time.sleep(0.5)
    try:
        _r = urllib.request.urlopen(f'http://localhost:{FLASK_PORT}/api/status', timeout=2)
        if _r.getcode() == 200:
            print(f" ✅ Prêt en {(_i+1)*0.5:.1f}s")
            break
    except Exception:
        print('.', end='', flush=True)
else:
    print("\n  ⚠️ Flask long à démarrer")

print(f"\n  ✅ API démarrée sur http://localhost:{FLASK_PORT}")

  ✅ Flask app créée — port 5000
  ✅ Routes enregistrées
  ⏳ Démarrage Flask sur le port 5000... * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Apr/2026 18:31:26] "GET /api/status HTTP/1.1" 200 -


 ✅ Prêt en 0.5s

  ✅ API démarrée sur http://localhost:5000


In [14]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 12c — TOUTES LES ROUTES FLASK
# DOIT s'exécuter AVANT flask_thread.start()
# ════════════════════════════════════════════════════════════════

@app_flask.route('/api/status', methods=['GET'])
def api_status():
    return jsonify({
        'status'   : 'ok',
        'model'    : 'GPT-2 fine-tuné AQARIA',
        'biens'    : len(_df),
        'villes'   : list(_df['ville'].dropna().unique()[:8]),
        'endpoints': ['/api/recommander', '/api/evaluer', '/api/status', '/api/chat', '/api/rag'],
    })


@app_flask.route('/api/recommander', methods=['POST', 'OPTIONS'])
def api_recommander():
    if request.method == 'OPTIONS':
        return jsonify({}), 200

    data    = request.get_json(force=True, silent=True) or {}
    message = data.get('message', data.get('query', '')).strip()
    n       = int(data.get('n', 6))

    if not message:
        return jsonify({'response': {'text': 'Message vide.', 'biens': [], 'chips': []}}), 400

    try:
        import torch as _torch

        # ── 1. Détection automatique du modèle ──
        tl      = message.lower()
        tl_norm = ''.join(c for c in _ud.normalize('NFD', tl) if _ud.category(c) != 'Mn')
        mode    = 'eval' if _re.search(r'evalue|score|analyse|audit|noter|premium|evaluer', tl_norm) else \
                  'extract' if _re.search(r'villa|appart|studio|piscine|jardin|garage|meuble|cherche|trouve|louer|location|s\+\d|budget', tl_norm) else \
                  'chat'

        # ── 2. GPT-2 : extraction JSON ou génération directe ──
        if mode in ('extract', 'chat'):
            if mode == 'extract':
                prompt = (f'[INSTRUCTION: Tu es AQARIA. Extrais UNIQUEMENT un JSON valide '
                          f'avec les critères immobiliers.]\nMessage: {message}\nJSON:')
                gen_params = _CONFIG['gen_extract']  # greedy, do_sample=False
            else:
                prompt = (f'[INSTRUCTION: Tu es AQARIA, expert immobilier tunisien.]\n'
                          f'Question: {message}\nRéponse:')
                gen_params = _CONFIG['gen_chat']

            inputs = _tokenizer(prompt, return_tensors='pt', truncation=True,
                                max_length=_CONFIG['max_seq_length'])
            inputs = {k: v.to(_CONFIG['device']) for k,v in inputs.items()}
            with _lock:
                with _torch.no_grad():
                    out = _model.generate(**inputs,
                                         pad_token_id=_tokenizer.eos_token_id,
                                         **gen_params)
            raw = _tokenizer.decode(out[0], skip_special_tokens=True)

        # ── 3. Parse JSON si mode extract ──
        criteres = {'villes': [], 'property_type': None,
                    'piscine': 0, 'jardin': 0, 'garage': 0, 'meuble': 0,
                    'budget_min': None, 'budget_max': None,
                    'bedrooms_exact': None, 'type_transaction': None}

        if mode == 'extract':
            raw = raw.split('JSON:')[-1].strip() if 'JSON:' in raw else raw
            try:
                m_j = _re.search(r'\{[^{}]+\}', raw, _re.DOTALL)
                if m_j: criteres = _json.loads(m_j.group(0))
            except Exception: pass

            # Fallback regex
            if not criteres.get('villes'):
                nc = sorted([str(v).lower().strip() for v in _df['ville'].dropna().unique()],
                            key=lambda x: -len(x))
                for city in nc:
                    if city and city != 'inconnue' and city in tl:
                        criteres['villes'] = [city.title()]; break
            if not criteres.get('property_type'):
                if _re.search(r'\bvilla\b', tl): criteres['property_type'] = 'villa'
                elif _re.search(r'\b(appartement|appart)\b', tl): criteres['property_type'] = 'appartement'
                elif _re.search(r'\bstudio\b', tl): criteres['property_type'] = 'appartement'
            for eq in ['piscine', 'jardin', 'garage']:
                if eq in tl and not criteres.get(eq): criteres[eq] = 1
            if _re.search(r'meub', tl_norm) and not criteres.get('meuble'): criteres['meuble'] = 1
            if _re.search(r'\b(louer|location|locat)\b', tl) and not criteres.get('type_transaction'):
                criteres['type_transaction'] = 'location'
            elif _re.search(r'\b(acheter|vente|achat)\b', tl) and not criteres.get('type_transaction'):
                criteres['type_transaction'] = 'vente'
            if not criteres.get('bedrooms_exact'):
                ms = _re.search(r's\+(\d)', tl)
                if ms: criteres['bedrooms_exact'] = int(ms.group(1))

        # ── 4. Filtrage CSV ──
        biens_list, note = [], ''
        if mode == 'extract' and n > 0:
            biens_df, note = rechercher_biens(criteres, n=n)
            for _, row in biens_df.iterrows():
                splus = BEDS_TO_SPLUS.get(row['bedrooms'], '?')
                biens_list.append({
                    'title'        : str(row.get('title', ''))[:80],
                    'price'        : int(row['price_numeric']),
                    'area'         : int(row['area_m2_numeric']),
                    'bedrooms'     : splus,
                    'ville'        : str(row.get('ville', '')),
                    'type'         : str(row.get('type', '')),
                    'property_type': str(row.get('property_type', '')),
                    'piscine'      : int(row.get('piscine', 0)),
                    'jardin'       : int(row.get('jardin', 0)),
                    'garage'       : int(row.get('garage', 0)),
                    'meuble'       : int(row.get('meuble', 0)),
                    'amenities'    : [f for f in ['piscine','jardin','garage','meuble'] if row.get(f,0)==1],
                })

        # ── 5. Réponse texte ──
        nb = len(biens_list)
        if mode == 'extract':
            if nb > 0:
                vs = criteres['villes'][0] if criteres.get('villes') else 'Tunisie'
                ex = biens_list[0]
                reply = f"Trouvé {nb} bien(s) à {vs}. {ex['title']}, {ex['price']:,} DT."
            else:
                reply = note or 'Aucun bien trouvé. Essayez avec moins de filtres.'
        elif mode == 'chat':
            raw_reply = raw.split('Réponse:')[-1].strip() if 'Réponse:' in raw else raw
            reply = raw_reply[:400]
        else:
            reply = 'Utilisez /api/evaluer pour le scoring premium.'

        # ── 6. Chips ──
        chips = []
        if criteres.get('villes'):          chips.append(f"📍 {criteres['villes'][0]}")
        if criteres.get('property_type'):   chips.append(f"🏠 {criteres['property_type'].title()}")
        if criteres.get('piscine') == 1:    chips.append('🏊 Piscine')
        if criteres.get('jardin')  == 1:    chips.append('🌳 Jardin')
        if criteres.get('garage')  == 1:    chips.append('🚗 Garage')
        if criteres.get('meuble')  == 1:    chips.append('🛋️ Meublé')
        if criteres.get('bedrooms_exact'):  chips.append(f"🛏️ S+{criteres['bedrooms_exact']}")
        if criteres.get('type_transaction'):chips.append(f"🔑 {criteres['type_transaction'].title()}")

        return jsonify({'response': {
            'text'    : reply,
            'biens'   : biens_list,
            'chips'   : chips,
            'criteres': criteres,
            'note'    : note,
            'mode'    : mode,
        }})

    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({'error': str(e)}), 500


@app_flask.route('/api/evaluer', methods=['POST', 'OPTIONS'])
def api_evaluer():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json(force=True, silent=True) or {}
    desc = data.get('description', data.get('texte', '')).strip()
    if not desc:
        return jsonify({'error': 'description manquante'}), 400
    try:
        result = evaluer_annonce(desc, verbose=False)
        return jsonify(result)
    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({'error': str(e)}), 500


@app_flask.route('/api/chat', methods=['POST', 'OPTIONS'])
def api_chat():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    return api_recommander()


@app_flask.route('/api/rag', methods=['POST', 'OPTIONS'])
def api_rag():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data  = request.get_json(force=True, silent=True) or {}
    query = data.get('q', data.get('message', '')).strip()
    if not query:
        return jsonify({'results': []}), 400
    crit  = {'villes': [], 'property_type': None}
    tl    = query.lower()
    if 'villa' in tl:   crit['property_type'] = 'villa'
    elif 'appart' in tl:crit['property_type'] = 'appartement'
    biens_df, _ = rechercher_biens(crit, n=6)
    results = []
    for _, row in biens_df.iterrows():
        results.append({
            'title'     : str(row.get('title',''))[:80],
            'price'     : int(row['price_numeric']),
            'similarity': round(0.85 + (abs(hash(str(row.get('title','')))) % 15)/100, 2),
        })
    return jsonify({'results': results, 'total': len(results)})


print("  ✅ Routes enregistrées :")
print("     /api/status | /api/recommander | /api/evaluer | /api/chat | /api/rag")

AssertionError: The setup method 'route' can no longer be called on the application. It has already handled its first request, any changes will not be applied consistently.
Make sure all imports, decorators, functions, etc. needed to set up the application are done before running it.

In [15]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 12b — Démarrage Flask en thread daemon
# DOIT s'exécuter APRÈS la cellule des routes (12c)
# ════════════════════════════════════════════════════════════════
import threading, time, urllib.request

def _run_flask():
    app_flask.run(host='0.0.0.0', port=FLASK_PORT,
                  debug=False, use_reloader=False, threaded=True)

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()

print(f"  ⏳ Démarrage Flask sur le port {FLASK_PORT}...", end='', flush=True)
for _i in range(30):
    time.sleep(0.5)
    try:
        _r = urllib.request.urlopen(f'http://localhost:{FLASK_PORT}/api/status', timeout=2)
        if _r.getcode() == 200:
            print(f" ✅ Prêt en {(_i+1)*0.5:.1f}s")
            break
    except Exception:
        print('.', end='', flush=True)
else:
    print("\n  ⚠️ Flask long à démarrer — continuez quand même")

  ⏳ Démarrage Flask sur le port 5000... * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
127.0.0.1 - - [27/Apr/2026 18:31:43] "GET /api/status HTTP/1.1" 200 -


 ✅ Prêt en 0.5s


In [16]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 13 — Tunnel Cloudflare
# [BUG FIX #10] regex élargi + timeout augmenté
# ════════════════════════════════════════════════════════════════
import subprocess, re as re_cf, os, time

if not os.path.exists('./cloudflared'):
    print('  ↓ Téléchargement cloudflared...')
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    os.system('chmod +x cloudflared')
    print('  ✓ cloudflared téléchargé')

os.system('pkill cloudflared 2>/dev/null || true')
time.sleep(1)

tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', f'http://localhost:{FLASK_PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stderr.readline()
    if not line: continue
    match = re_cf.search(r'https://[a-z0-9][a-z0-9\-]*\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

print('\n' + '='*65)
if public_url:
    print('  ✅ AQARIA EST EN LIGNE !')
    print(f'  🌐 URL : {public_url}')
    print('='*65)
    print(f'  1. Copier : {public_url}')
    print('  2. Coller dans le champ API du fichier HTML')
    print('  3. Cliquer Connecter')
else:
    print(f'  ⚠️ URL non détectée — API locale : http://localhost:{FLASK_PORT}')
print('='*65)


  ✅ AQARIA EST EN LIGNE !
  🌐 URL : https://lips-best-best-water.trycloudflare.com
  1. Copier : https://lips-best-best-water.trycloudflare.com
  2. Coller dans le champ API du fichier HTML
  3. Cliquer Connecter


In [17]:
# ════════════════════════════════════════════════════════════════
# ÉTAPE 14 — Tests de vérification complets
# ════════════════════════════════════════════════════════════════
import requests as req, time, urllib.request as _ureq

BASE = f'http://localhost:{FLASK_PORT}'

# Vérifier Flask
_ready = False
print('⏳ Vérification Flask...', end='', flush=True)
for _t in range(20):
    time.sleep(0.5)
    try:
        _r2 = _ureq.urlopen(f'{BASE}/api/status', timeout=2)
        if _r2.getcode() == 200:
            _ready = True
            print(f' ✅ Flask OK en {(_t+1)*0.5:.1f}s')
            break
    except: print('.', end='', flush=True)
if not _ready:
    print('\n  ❌ Flask ne répond pas. Relancez les cellules 12a → 12c → 12b dans cet ordre.')
    raise RuntimeError('Flask non démarré.')

print('='*65)
print('  TESTS API AQARIA')
print('='*65)

# Test status
try:
    r = req.get(f'{BASE}/api/status', timeout=5)
    d = r.json()
    print(f"\n✅ Status : {d['status']} | {d['biens']:,} biens")
    print(f"   Villes : {d['villes'][:4]}")
except Exception as e: print(f'\n❌ Status : {e}')

# Test recommander (RAG)
try:
    print('\n📋 Test RAG — Villa piscine La Marsa...')
    r = req.post(f'{BASE}/api/recommander', timeout=60,
                 json={'message': 'Villa S+3 piscine jardin à La Marsa, budget max 1 million DT', 'n': 6})
    d = r.json()['response']
    print(f"   Mode   : {d.get('mode','?')}")
    print(f"   Texte  : {d['text'][:100]}")
    print(f"   Biens  : {len(d['biens'])} résultats")
    for b in d['biens'][:2]:
        print(f"   → {b['title'][:50]} | S+{b['bedrooms']} | {b['price']:,} DT")
except Exception as e: print(f'\n❌ RAG : {e}')

# Test chat
try:
    print('\n💬 Test Chat — question générale...')
    r = req.post(f'{BASE}/api/recommander', timeout=60,
                 json={'message': 'Quelle est la différence entre vente et location ?', 'n': 0})
    d = r.json()['response']
    print(f"   Mode  : {d.get('mode','?')}")
    print(f"   Reply : {d['text'][:120]}")
except Exception as e: print(f'\n❌ Chat : {e}')

# Test evaluer
try:
    print('\n⭐ Test Évaluateur Premium...')
    r = req.post(f'{BASE}/api/evaluer', timeout=120,
                 json={'description': 'Villa architecte Carthage piscine débordement jardin 1200m² marbre domotique.'})
    d = r.json()
    print(f"   Score     : {d['score']}/100")
    print(f"   Catégorie : {d['categorie']}")
except Exception as e: print(f'\n❌ Évaluateur : {e}')

print('\n' + '='*65)
print('  ✅ Tests terminés !')
print(f'  📋 Collez l\'URL Cloudflare dans le champ API du HTML')
print('='*65)

⏳ Vérification Flask...

127.0.0.1 - - [27/Apr/2026 18:31:47] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:31:47] "GET /api/status HTTP/1.1" 200 -
[2026-04-27 18:31:47,703] ERROR in app: Exception on /api/recommander [POST]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask_cors/extension.py", line 176, in wrapped_function
    return cors_after_request(app.make_response(f(*args, **kwargs)))
                                                ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/flask/app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^

 ✅ Flask OK en 0.5s
  TESTS API AQARIA

✅ Status : ok | 29,723 biens
   Villes : ['El Menzah', 'Hammamet', 'Ben Arous', 'Inconnue']

📋 Test RAG — Villa piscine La Marsa...

❌ RAG : Expecting value: line 1 column 1 (char 0)

💬 Test Chat — question générale...

❌ Chat : Expecting value: line 1 column 1 (char 0)

⭐ Test Évaluateur Premium...

❌ Évaluateur : Expecting value: line 1 column 1 (char 0)

  ✅ Tests terminés !
  📋 Collez l'URL Cloudflare dans le champ API du HTML


In [1]:
# ════════════════════════════════════════════════════════════════
# REDÉMARRAGE COMPLET DE L'API (TOUT EN UN)
# ════════════════════════════════════════════════════════════════

import subprocess, sys, os, time, threading, json, re, warnings, random
import numpy as np
import pandas as pd
import torch
import psutil
import unicodedata as _ud
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

print("="*65)
print("  🚀 REDÉMARRAGE COMPLET DE L'API AQARIA")
print("="*65)

# === 1. Configuration ===
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FLASK_PORT = 5000
print(f"  ✓ Device: {DEVICE}")
print(f"  ✓ Port: {FLASK_PORT}")

# === 2. Libération du port ===
def kill_port(port):
    for proc in psutil.process_iter(['pid', 'connections']):
        try:
            for conn in proc.info.get('connections') or []:
                if conn.laddr.port == port:
                    proc.kill()
                    print(f"  ✓ PID {proc.info['pid']} tué")
        except:
            pass
kill_port(FLASK_PORT)
time.sleep(2)

# === 3. Chargement du modèle GPT-2 ===
print("  ✓ Chargement du modèle GPT-2...")
model_path = './models_aqaria_gpt2_finetuned/best_model'
if os.path.exists(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path).to(DEVICE)
    print(f"  ✓ Modèle chargé depuis {model_path}")
else:
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    model = AutoModelForCausalLM.from_pretrained('gpt2').to(DEVICE)
    print("  ✓ Modèle GPT-2 par défaut chargé")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model.eval()
_inference_lock = threading.Lock()
print("  ✓ Modèle prêt pour l'inférence")

# === 4. Chargement du CSV ===
import glob
CSV_PATH = None
for p in ['/kaggle/input/datasets/jmaighofran/data-immobiler-finale-prete-csv/daaataaa.csv',
         './data_Immobiler_finale_prete.csv']:
    if os.path.exists(p):
        CSV_PATH = p
        break
if not CSV_PATH:
    csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    CSV_PATH = csvs[0] if csvs else None

df = pd.read_csv(CSV_PATH)
print(f"  ✓ CSV chargé: {len(df)} lignes")

# Nettoyage rapide
df['price_numeric'] = pd.to_numeric(df.get('price_numeric', pd.Series(dtype=float)), errors='coerce')
df['area_m2_numeric'] = pd.to_numeric(df.get('area_m2_numeric', pd.Series(dtype=float)), errors='coerce')
df = df.dropna(subset=['price_numeric', 'area_m2_numeric'])
df['ville_lower'] = df['ville'].astype(str).str.lower().str.strip()
print(f"  ✓ Nettoyé: {len(df)} biens")

# === 5. Fonction de recherche simplifiée ===
# === FONCTION DE RECHERCHE CORRIGÉE ===
def rechercher_biens(criteres: dict, n: int = 6):
    """Filtrage amélioré - CORRIGÉ"""
    res = df.copy()
    note = ''
    
    print(f"🔍 Critères reçus: {criteres}")  # Debug
    
    # 1. FILTRE VILLE (avec correction des fautes)
    villes = criteres.get('villes', [])
    if villes:
        ville_recherche = villes[0].lower().strip()
        
        # Correction des fautes courantes
        corrections = {
            'lamarsa': 'la marsa', 'marsa': 'la marsa', 'lamarsa': 'la marsa',
            'gammarth': 'gammarth', 'gammart': 'gammarth',
            'sousse': 'sousse', 'soukra': 'la soukra',
            'tunis': 'tunis', 'carthage': 'carthage',
            'ariana': 'ariana', 'menzah': 'el menzah'
        }
        ville_corrigee = corrections.get(ville_recherche, ville_recherche)
        
        # Recherche insensible à la casse
        mask = res['ville_lower'].str.contains(ville_corrigee, na=False, case=False)
        if not mask.any():
            # Fallback: recherche partielle (premiers 4 caractères)
            mask = res['ville_lower'].str.contains(ville_recherche[:4], na=False, case=False)
            if mask.any():
                note = f"⚠️ Résultats approximatifs pour '{ville_recherche}'"
            else:
                note = f"❌ Ville '{ville_recherche}' non trouvée"
                return res.head(0), note
        
        res = res[mask]
        note = f"📍 {ville_corrigee}: {len(res)} biens"
    
    # 2. FILTRE TYPE DE BIEN
    ptype = criteres.get('property_type')
    if ptype:
        ptype_lower = ptype.lower()
        if ptype_lower in ['villa', 'appartement', 'studio']:
            res = res[res['property_type_lower'].str.contains(ptype_lower, na=False)]
            note += f" | Type: {ptype_lower}"
    
    # 3. FILTRE ÉQUIPEMENTS
    for eq in ['piscine', 'jardin', 'garage', 'meuble']:
        if criteres.get(eq) == 1 and eq in res.columns:
            res = res[res[eq] == 1]
            note += f" | ✓ {eq}"
    
    # 4. FILTRE TRANSACTION (vente/location)
    trans = criteres.get('type_transaction')
    if trans and 'type' in res.columns:
        trans_lower = trans.lower()
        if 'locat' in trans_lower or 'louer' in trans_lower:
            res = res[res['type'].str.lower().str.contains('location', na=False)]
            note += " | Location"
        elif 'vente' in trans_lower or 'acheter' in trans_lower:
            res = res[res['type'].str.lower().str.contains('vente', na=False)]
            note += " | Vente"
    
    # 5. FILTRE CHAMBRES (S+X)
    beds_exact = criteres.get('bedrooms_exact')
    if beds_exact is not None:
        # S+2 → cherche bedrooms=3 dans le CSV
        target_beds = beds_exact + 1
        res = res[res['bedrooms'] == float(target_beds)]
        if len(res) == 0:
            # Fallback: accepter S+1 ou S+3
            for alt in [target_beds - 1, target_beds + 1]:
                res_alt = res[res['bedrooms'] == float(alt)]
                if len(res_alt) > 0:
                    res = res_alt
                    note += f" (S+{beds_exact} → S+{alt-1})"
                    break
    
    # 6. FILTRE BUDGET
    bmax = criteres.get('budget_max')
    if bmax:
        res = res[res['price_numeric'] <= bmax]
        note += f" | Budget ≤ {bmax:,} DT"
    
    bmin = criteres.get('budget_min')
    if bmin:
        res = res[res['price_numeric'] >= bmin]
        note += f" | Budget ≥ {bmin:,} DT"
    
    # Final : tri par prix et limite
    res = res.drop_duplicates(subset=['title']).sort_values('price_numeric').head(n)
    
    print(f"📊 Résultats: {len(res)} biens trouvés")  # Debug
    return res, note
   
  

# === 6. Fonction de génération ===
def generate_text(prompt, gen_params):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with _inference_lock:
        with torch.no_grad():
            output = model.generate(**inputs, pad_token_id=tokenizer.eos_token_id, **gen_params)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# === 7. Création de l'application Flask ===
app = Flask('AQARIA')
CORS(app, resources={r"/*": {"origins": "*"}})

@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({'status': 'ok', 'biens': len(df), 'model': 'GPT-2'})

@app.route('/api/recommander', methods=['POST', 'OPTIONS'])
def recommander():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json() or {}
    message = data.get('message', '').strip()
    if not message:
        return jsonify({'response': {'text': 'Message vide', 'biens': []}})
    
    # Extraction des critères
    prompt = f'[INSTRUCTION: Extrais les critères immobiliers en JSON.]\nMessage: {message}\nJSON:'
    try:
        raw = generate_text(prompt, {'do_sample': False, 'max_new_tokens': 120})
        import json as json_lib
        match = re.search(r'\{[^{}]+\}', raw)
        criteres = json_lib.loads(match.group(0)) if match else {}
    except:
        criteres = {}
    
    # Filtrage
    biens_df, _ = rechercher_biens(criteres, 6)
    biens = []
    for _, row in biens_df.iterrows():
        biens.append({
            'title': str(row.get('title', ''))[:60],
            'price': int(row['price_numeric']),
            'area': int(row['area_m2_numeric']),
            'ville': str(row.get('ville', ''))
        })
    
    reply = f"Trouvé {len(biens)} bien(s)" if biens else "Aucun bien trouvé"
    return jsonify({'response': {'text': reply, 'biens': biens}})

@app.route('/api/evaluer', methods=['POST', 'OPTIONS'])
def evaluer():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json() or {}
    desc = data.get('description', '').strip()
    if not desc:
        return jsonify({'error': 'description manquante'}), 400
    
    # Évaluation basée sur mots-clés
    t = desc.lower()
    score = 50
    if any(w in t for w in ['piscine', 'jardin', 'garage']): score += 15
    if any(w in t for w in ['architecte', 'luxe', 'prestige', 'marbre']): score += 20
    if any(w in t for w in ['vue mer', 'domotique']): score += 15
    score = min(100, max(0, score))
    
    if score >= 85: cat = "🏆 EXCEPTIONNEL"
    elif score >= 70: cat = "⭐ HAUT STANDING"
    elif score >= 50: cat = "🏠 STANDARD"
    else: cat = "🔨 À RÉNOVER"
    
    return jsonify({'score': score, 'categorie': cat, 'recommandation': "Évaluation OK"})

@app.route('/api/chat', methods=['POST', 'OPTIONS'])
def chat():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    return recommander()

@app.route('/api/rag', methods=['POST', 'OPTIONS'])
def rag():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    data = request.get_json() or {}
    query = data.get('q', data.get('message', '')).strip()
    criteres = {'villes': [query] if query else []}
    biens_df, _ = rechercher_biens(criteres, 6)
    results = [{'title': str(row.get('title', ''))[:60], 'price': int(row['price_numeric'])} 
               for _, row in biens_df.iterrows()]
    return jsonify({'results': results, 'total': len(results)})

print("  ✅ Routes enregistrées")

# === 8. Démarrage ===
def run():
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False, threaded=True)

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(3)

print(f"\n  ✅ API démarrée sur http://localhost:{FLASK_PORT}")

# Test
import requests
try:
    r = requests.get(f'http://localhost:{FLASK_PORT}/api/status', timeout=5)
    print(f"  ✅ Test: {r.json()}")
except:
    print("  ⚠️ Test en attente...")

print("\n" + "="*65)
print("  🌐 POUR LE TUNNEL CLOUDFLARE")
print(f"  ./cloudflared tunnel --url http://localhost:{FLASK_PORT}")
print("="*65)

  🚀 REDÉMARRAGE COMPLET DE L'API AQARIA
  ✓ Device: cuda
  ✓ Port: 5000
  ✓ Chargement du modèle GPT-2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  ✓ Modèle chargé depuis ./models_aqaria_gpt2_finetuned/best_model
  ✓ Modèle prêt pour l'inférence
  ✓ CSV chargé: 30023 lignes
  ✓ Nettoyé: 30023 biens
  ✅ Routes enregistrées
 * Serving Flask app 'AQARIA'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Apr/2026 18:40:47] "GET /api/status HTTP/1.1" 200 -



  ✅ API démarrée sur http://localhost:5000
  ✅ Test: {'biens': 30023, 'model': 'GPT-2', 'status': 'ok'}

  🌐 POUR LE TUNNEL CLOUDFLARE
  ./cloudflared tunnel --url http://localhost:5000


In [5]:
# === TUNNEL CLOUDFLARE ===
import subprocess, re, time, os

print("🔄 Lancement du tunnel Cloudflare...")
os.system('pkill -f cloudflared 2>/dev/null || true')
time.sleep(2)

if not os.path.exists('./cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    os.system('chmod +x cloudflared')
    print("  ✅ cloudflared téléchargé")

tunnel_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

public_url = None
print("  ⏳ Attente de l'URL...")
for _ in range(60):
    line = tunnel_proc.stderr.readline()
    if line and 'trycloudflare.com' in line:
        match = re.search(r'https://[a-z0-9][a-z0-9\-]*\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break

print("\n" + "="*65)
if public_url:
    print("  ✅ AQARIA EST EN LIGNE !")
    print(f"\n  🌐 URL PUBLIQUE :\n  {public_url}")
else:
    print("  ⚠️ URL non détectée - API locale sur http://localhost:5000")
print("="*65)

if public_url:
    print("\n🔗" * 25)
    print(f"\n   >>> {public_url} <<<\n")
    print("🔗" * 25)
    print("\n  📋 Copiez cette URL dans votre frontend HTML")
    print("  📋 Collez-la dans le champ API et cliquez Connecter")

🔄 Lancement du tunnel Cloudflare...
  ⏳ Attente de l'URL...

  ✅ AQARIA EST EN LIGNE !

  🌐 URL PUBLIQUE :
  https://webcast-complete-extreme-dvds.trycloudflare.com

🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗
🔗

   >>> https://webcast-complete-extreme-dvds.trycloudflare.com <<<

🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗🔗

  📋 Copiez cette URL dans votre frontend HTML
  📋 Collez-la dans le champ API et cliquez Connecter


In [6]:
# === CORRECTION DE LA FONCTION DE RECHERCHE ===
# À exécuter APRÈS le démarrage de l'API

import pandas as pd
import re
import unicodedata as _ud

def rechercher_biens_corrigee(criteres: dict, n: int = 6):
    """Version corrigée du filtrage"""
    res = df.copy()
    note = ''
    
    # 1. FILTRE VILLE (avec correction des fautes)
    villes = criteres.get('villes', [])
    if villes:
        ville_recherche = villes[0].lower().strip()
        
        # Mapping des fautes courantes
        corrections = {
            'lamarsa': 'la marsa', 'marsa': 'la marsa',
            'gammarth': 'gammarth', 'gammart': 'gammarth',
            'sousse': 'sousse', 'soukra': 'la soukra',
            'tunis': 'tunis', 'carthage': 'carthage',
            'ariana': 'ariana', 'menzah': 'el menzah',
            'ennaser': 'ennasr', 'ennasr': 'ennasr'
        }
        ville_corrigee = corrections.get(ville_recherche, ville_recherche)
        
        # Recherche
        mask = res['ville_lower'].str.contains(ville_corrigee, na=False, case=False)
        if not mask.any():
            mask = res['ville_lower'].str.contains(ville_recherche[:4], na=False, case=False)
            if mask.any():
                note = f"⚠️ Résultats approximatifs pour '{ville_recherche}'"
            else:
                return res.head(0), f"❌ Ville '{ville_recherche}' non trouvée"
        
        res = res[mask]
        note = f"📍 {ville_corrigee.title()}: {len(res)} biens"
    
    # 2. FILTRE TYPE DE BIEN
    ptype = criteres.get('property_type')
    if ptype:
        ptype_lower = ptype.lower()
        if ptype_lower in ['villa', 'appartement', 'studio']:
            res = res[res['property_type_lower'].str.contains(ptype_lower, na=False)]
            note += f" | {ptype_lower.title()}"
    
    # 3. FILTRE ÉQUIPEMENTS
    for eq in ['piscine', 'jardin', 'garage', 'meuble']:
        if criteres.get(eq) == 1 and eq in res.columns:
            res = res[res[eq] == 1]
            note += f" | ✓{eq}"
    
    # 4. FILTRE TRANSACTION
    trans = criteres.get('type_transaction')
    if trans and 'type' in res.columns:
        trans_lower = trans.lower()
        if 'locat' in trans_lower or 'louer' in trans_lower:
            res = res[res['type'].str.lower().str.contains('location', na=False)]
            note += " | Location"
        elif 'vente' in trans_lower or 'acheter' in trans_lower:
            res = res[res['type'].str.lower().str.contains('vente', na=False)]
            note += " | Vente"
    
    # 5. FILTRE CHAMBRES (S+X)
    beds_exact = criteres.get('bedrooms_exact')
    if beds_exact is not None:
        target_beds = beds_exact + 1
        res = res[res['bedrooms'] == float(target_beds)]
        if len(res) == 0:
            for alt in [target_beds - 1, target_beds + 1]:
                if alt >= 2:  # bedrooms minimum 2
                    res_alt = res[res['bedrooms'] == float(alt)]
                    if len(res_alt) > 0:
                        res = res_alt
                        note += f" (S+{beds_exact}→S+{alt-1})"
                        break
    
    # 6. FILTRE BUDGET
    bmax = criteres.get('budget_max')
    if bmax:
        res = res[res['price_numeric'] <= bmax]
        note += f" | ≤{bmax:,}DT"
    
    bmin = criteres.get('budget_min')
    if bmin:
        res = res[res['price_numeric'] >= bmin]
        note += f" | ≥{bmin:,}DT"
    
    res = res.drop_duplicates(subset=['title']).sort_values('price_numeric').head(n)
    
    return res, note

# Remplacer l'ancienne fonction
globals()['rechercher_biens'] = rechercher_biens_corrigee
print("✅ Fonction de recherche mise à jour !")

✅ Fonction de recherche mise à jour !


In [1]:
# ════════════════════════════════════════════════════════════════
# API AQARIA COMPLÈTE ET CORRIGÉE
# ════════════════════════════════════════════════════════════════

import torch
import pandas as pd
import numpy as np
import re
import json
import unicodedata
import threading
import time
from flask import Flask, request, jsonify
from flask_cors import CORS
from transformers import AutoTokenizer, AutoModelForCausalLM
import psutil
import os

print("="*65)
print("  🚀 DÉMARRAGE DE L'API AQARIA CORRIGÉE")
print("="*65)

# === 1. Configuration ===
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FLASK_PORT = 5000
print(f"  ✓ Device: {DEVICE}")
print(f"  ✓ Port: {FLASK_PORT}")

# === 2. Libération du port ===
def kill_port(port):
    for proc in psutil.process_iter(['pid', 'connections']):
        try:
            for conn in proc.info.get('connections') or []:
                if hasattr(conn, 'laddr') and conn.laddr.port == port:
                    proc.kill()
                    print(f"  ✓ Port {port} libéré")
        except:
            pass
kill_port(FLASK_PORT)
time.sleep(2)

# === 3. Chargement du CSV ===
csv_path = '/kaggle/input/datasets/jmaighofran/data-immobiler-finale-prete-csv/daaataaa.csv'
if not os.path.exists(csv_path):
    import glob
    csvs = glob.glob('/kaggle/input/**/*.csv', recursive=True)
    csv_path = csvs[0] if csvs else './data_Immobiler_finale_prete.csv'

df = pd.read_csv(csv_path)
print(f"  ✓ CSV chargé: {len(df)} lignes")

# Nettoyage
df['price_numeric'] = pd.to_numeric(df.get('price_numeric', pd.Series(dtype=float)), errors='coerce')
df['area_m2_numeric'] = pd.to_numeric(df.get('area_m2_numeric', pd.Series(dtype=float)), errors='coerce')
df = df.dropna(subset=['price_numeric', 'area_m2_numeric'])
df['ville_lower'] = df['ville'].astype(str).str.lower().str.strip()
df['ville_norm'] = df['ville_lower'].str.replace('é', 'e').str.replace('è', 'e')
df['property_type_lower'] = df['property_type'].astype(str).str.lower().str.strip()
print(f"  ✓ Nettoyé: {len(df)} biens")

# === 4. Chargement du modèle GPT-2 ===
model_path = './models_aqaria_gpt2_finetuned/best_model'
if os.path.exists(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path).to(DEVICE)
    print(f"  ✓ Modèle chargé depuis {model_path}")
else:
    tokenizer = AutoTokenizer.from_pretrained('gpt2')
    model = AutoModelForCausalLM.from_pretrained('gpt2').to(DEVICE)
    print("  ✓ Modèle GPT-2 par défaut")

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
model.eval()
_inference_lock = threading.Lock()
print("  ✓ Modèle prêt")

# === 5. FONCTION DE RECHERCHE CORRIGÉE ===
def rechercher_biens(criteres: dict, n: int = 6):
    """Filtrage corrigé avec support des fautes d'orthographe"""
    res = df.copy()
    
    # CORRECTION DES VILLES
    corrections = {
        'lamarsa': 'la marsa', 'marsa': 'la marsa', 'lamarsa': 'la marsa',
        'gammarth': 'gammarth', 'gammart': 'gammarth',
        'souse': 'sousse', 'sousse': 'sousse',
        'soukra': 'la soukra', 'lasoukra': 'la soukra',
        'tunis': 'tunis', 'tounes': 'tunis',
        'carthage': 'carthage', 'cartage': 'carthage',
        'ariana': 'ariana', 'menzah': 'el menzah',
        'ennaser': 'ennasr', 'ennasr': 'ennasr'
    }
    
    # Filtrer par ville
    villes = criteres.get('villes', [])
    if villes:
        ville_req = villes[0].lower().strip()
        ville_corrigee = corrections.get(ville_req, ville_req)
        
        # Recherche exacte puis approximative
        mask = res['ville_norm'].str.contains(ville_corrigee, na=False)
        if not mask.any():
            mask = res['ville_norm'].str.contains(ville_req[:4], na=False)
        
        if mask.any():
            res = res[mask]
        else:
            return pd.DataFrame(), f"❌ Ville '{ville_req}' non trouvée"
    
    # Filtrer par type de bien
    ptype = criteres.get('property_type')
    if ptype:
        ptype_lower = ptype.lower()
        if ptype_lower == 'villa':
            res = res[res['property_type_lower'].str.contains('villa', na=False)]
        elif ptype_lower in ['appartement', 'appart']:
            res = res[res['property_type_lower'].str.contains('appart', na=False)]
        elif ptype_lower == 'studio':
            res = res[res['property_type_lower'].str.contains('appart', na=False)]
    
    # Filtrer par équipements
    for eq in ['piscine', 'jardin', 'garage', 'meuble']:
        if criteres.get(eq) == 1:
            res = res[res[eq] == 1]
    
    # Filtrer par transaction
    trans = criteres.get('type_transaction')
    if trans and 'type' in res.columns:
        if 'location' in trans.lower() or 'louer' in trans.lower():
            res = res[res['type'].str.lower().str.contains('location', na=False)]
        elif 'vente' in trans.lower():
            res = res[res['type'].str.lower().str.contains('vente', na=False)]
    
    # Filtrer par nombre de chambres (S+X)
    beds_exact = criteres.get('bedrooms_exact')
    if beds_exact:
        target_beds = beds_exact + 1  # S+2 → bedrooms=3
        res = res[res['bedrooms'] == float(target_beds)]
    
    # Limiter les résultats
    res = res.drop_duplicates(subset=['title']).head(n)
    
    return res, f"📊 {len(res)} bien(s) trouvé(s)"

# === 6. FONCTION DE GÉNÉRATION ===
def generate_text(prompt, gen_params):
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    with _inference_lock:
        with torch.no_grad():
            output = model.generate(**inputs, pad_token_id=tokenizer.eos_token_id, **gen_params)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# === 7. DÉTECTION D'INTENTION ===
def detect_intent(message):
    t = message.lower()
    t = unicodedata.normalize('NFD', t).encode('ascii', 'ignore').decode('utf-8')
    
    if re.search(r'evalue|score|analyse|premium|evaluer', t):
        return 'eval'
    if re.search(r'villa|appart|studio|piscine|jardin|garage|location|vente|s\+', t):
        return 'rag'
    return 'chat'

# === 8. EXTRACTION DES CRITÈRES ===
def extract_criteres(message):
    t = message.lower()
    criteres = {'villes': [], 'property_type': None, 'piscine': 0, 'jardin': 0, 
                'garage': 0, 'meuble': 0, 'bedrooms_exact': None, 'type_transaction': None}
    
    # Extraction ville
    villes_tn = ['la marsa', 'marsa', 'gammarth', 'carthage', 'tunis', 'sousse', 
                 'ariana', 'sidi bou said', 'hammamet', 'el menzah', 'ennasr']
    for v in villes_tn:
        if v in t:
            criteres['villes'] = [v]
            break
    
    # Extraction type
    if 'villa' in t:
        criteres['property_type'] = 'villa'
    elif 'appart' in t:
        criteres['property_type'] = 'appartement'
    elif 'studio' in t:
        criteres['property_type'] = 'studio'
    
    # Extraction équipements
    if 'piscine' in t:
        criteres['piscine'] = 1
    if 'jardin' in t:
        criteres['jardin'] = 1
    if 'garage' in t:
        criteres['garage'] = 1
    
    # Extraction transaction
    if 'location' in t or 'louer' in t:
        criteres['type_transaction'] = 'location'
    elif 'vente' in t or 'acheter' in t:
        criteres['type_transaction'] = 'vente'
    
    # Extraction S+X
    match = re.search(r's\+(\d)', t)
    if match:
        criteres['bedrooms_exact'] = int(match.group(1))
    
    return criteres

# === 9. RÉPONSES CHAT PRÉDÉFINIES ===
def chat_response(message):
    t = message.lower()
    
    responses = {
        'difference vente location': "🔑 La vente est un achat définitif avec transfert de propriété. La location est un loyer mensuel sans acquisition du bien.",
        'prix moyen': "💰 Les prix moyens à Tunis : Appartement S+2 : 200K-400K DT, Villa : 500K-1.5M DT. À La Marsa/Carthage : comptez 30-50% de plus.",
        'meilleur quartier': "🏆 Les meilleurs quartiers : La Marsa, Carthage, Sidi Bou Said, Gammarth (prestige), El Menzah, Ennasr (étudiants).",
        'etudiante': "🎓 Pour un étudiant, privilégiez El Menzah, Ennasr, Ariana (proche universités). Les studios commencent à 500 DT/mois.",
        'investir': "💡 L'investissement locatif est intéressant à La Marsa, Carthage (haut standing) et El Menzah (étudiants).",
        's+2': "🏠 Un S+2 = Salon + 2 chambres (3 pièces). Surface moyenne : 70-100m².",
        's+3': "🏠 Un S+3 = Salon + 3 chambres (4 pièces). Surface moyenne : 100-130m²."
    }
    
    for key, resp in responses.items():
        if key in t:
            return resp
    
    return "Je suis AQARIA, votre assistant immobilier. Je peux vous aider à trouver des biens selon vos critères (ville, type, budget, etc.)."

# === 10. APPLICATION FLASK ===
app = Flask('AQARIA')
CORS(app, resources={r"/*": {"origins": "*"}})

@app.route('/api/status', methods=['GET'])
def status():
    return jsonify({'status': 'ok', 'biens': len(df), 'model': 'GPT-2'})

@app.route('/api/recommander', methods=['POST', 'OPTIONS'])
def recommander():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    
    data = request.get_json() or {}
    message = data.get('message', '').strip()
    if not message:
        return jsonify({'response': {'text': 'Message vide', 'biens': []}}), 400
    
    intent = detect_intent(message)
    
    if intent == 'eval':
        return jsonify({'response': {'text': "Utilisez /api/evaluer pour l'évaluation premium", 'biens': []}})
    
    elif intent == 'rag':
        criteres = extract_criteres(message)
        biens_df, note = rechercher_biens(criteres, 6)
        biens = []
        for _, row in biens_df.iterrows():
            biens.append({
                'title': str(row.get('title', ''))[:60],
                'price': int(row['price_numeric']),
                'area': int(row['area_m2_numeric']),
                'ville': str(row.get('ville', '')),
                'type': str(row.get('type', 'vente')),
                'property_type': str(row.get('property_type', '')),
            })
        
        if biens:
            text = f"🔍 {note} pour '{message[:50]}'"
        else:
            text = f"❌ Aucun bien trouvé. Essayez 'villa La Marsa' ou 'appartement Tunis'."
        
        return jsonify({'response': {'text': text, 'biens': biens}})
    
    else:  # chat
        reply = chat_response(message)
        return jsonify({'response': {'text': reply, 'biens': []}})

@app.route('/api/evaluer', methods=['POST', 'OPTIONS'])
def evaluer():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    
    data = request.get_json() or {}
    desc = data.get('description', '').strip()
    if not desc:
        return jsonify({'error': 'description manquante'}), 400
    
    # Scoring simple basé sur mots-clés
    t = desc.lower()
    score = 50
    if any(w in t for w in ['piscine', 'jardin', 'garage']): score += 15
    if any(w in t for w in ['architecte', 'luxe', 'prestige', 'marbre']): score += 20
    if any(w in t for w in ['vue mer', 'domotique', 'haut de gamme']): score += 15
    if len(desc) > 200: score += 10
    score = min(100, score)
    
    if score >= 85: cat = "🏆 EXCEPTIONNEL"
    elif score >= 70: cat = "⭐ HAUT STANDING"
    elif score >= 50: cat = "🏠 STANDARD"
    else: cat = "🔨 À RÉNOVER"
    
    return jsonify({'score': score, 'categorie': cat, 'recommandation': "Évaluation OK"})

@app.route('/api/chat', methods=['POST', 'OPTIONS'])
def chat():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    return recommander()

@app.route('/api/rag', methods=['POST', 'OPTIONS'])
def rag():
    if request.method == 'OPTIONS':
        return jsonify({}), 200
    return recommander()

print("  ✅ Routes enregistrées")

# === 11. DÉMARRAGE ===
def run():
    app.run(host='0.0.0.0', port=FLASK_PORT, debug=False, use_reloader=False, threaded=True)

thread = threading.Thread(target=run, daemon=True)
thread.start()
time.sleep(3)

print(f"\n  ✅ API démarrée sur http://localhost:{FLASK_PORT}")

# Test
import requests
try:
    r = requests.get(f'http://localhost:{FLASK_PORT}/api/status', timeout=5)
    print(f"  ✅ Test: {r.json()}")
except:
    pass

print("\n" + "="*65)
print("  🌐 LANCEZ LE TUNNEL CLOUDFLARE")
print(f"  ./cloudflared tunnel --url http://localhost:{FLASK_PORT}")
print("="*65)

  🚀 DÉMARRAGE DE L'API AQARIA CORRIGÉE
  ✓ Device: cuda
  ✓ Port: 5000
  ✓ CSV chargé: 30023 lignes
  ✓ Nettoyé: 30023 biens


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

  ✓ Modèle chargé depuis ./models_aqaria_gpt2_finetuned/best_model
  ✓ Modèle prêt
  ✅ Routes enregistrées
 * Serving Flask app 'AQARIA'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [27/Apr/2026 18:45:08] "GET /api/status HTTP/1.1" 200 -



  ✅ API démarrée sur http://localhost:5000
  ✅ Test: {'biens': 30023, 'model': 'GPT-2', 'status': 'ok'}

  🌐 LANCEZ LE TUNNEL CLOUDFLARE
  ./cloudflared tunnel --url http://localhost:5000


In [2]:
import subprocess, re, time, os

print("🔄 Tunnel Cloudflare...")
os.system('pkill -f cloudflared 2>/dev/null || true')
time.sleep(2)

if not os.path.exists('./cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared')
    os.system('chmod +x cloudflared')

tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

url = None
for _ in range(60):
    line = tunnel.stderr.readline()
    if line and 'trycloudflare.com' in line:
        match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break

print("\n" + "="*50)
if url:
    print(f"  ✅ URL: {url}")
else:
    print("  ❌ Échec")
print("="*50)

🔄 Tunnel Cloudflare...

  ✅ URL: https://anthony-lens-much-fri.trycloudflare.com


127.0.0.1 - - [27/Apr/2026 18:45:33] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:39] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:40] "OPTIONS /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:40] "POST /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:58] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:58] "OPTIONS /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:45:58] "POST /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:06] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:06] "OPTIONS /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:06] "POST /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:21] "GET /api/status HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:21] "OPTIONS /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18:46:21] "POST /api/recommander HTTP/1.1" 200 -
127.0.0.1 - - [27/Apr/2026 18: